# Task 2 — Leaderboard Challenge: Autoformer

**Goal.** Forecast `value` for `time_idx = 43657 … 43824` (exactly `pred_len = 168` steps) with an
**Autoformer** (Wu et al., 2021), and rank under `Score = RMSE + α·P + β·E`.

**Design summary**

| Decision | Choice | Why |
|---|---|---|
| Architecture | Autoformer encoder–decoder: progressive `SeriesDecomp` + FFT `AutoCorrelation` | hard requirement (§2.4.1); both mechanisms are genuinely used |
| Target space | `log1p` + global standardisation, with Duan smearing on inversion | series is non-negative and heavy-tailed (level moves an order of magnitude); MSE in raw space is dominated by a handful of spikes |
| Covariates | all 10 optional variables are **known across the horizon** → they enter the **decoder** stamp as well as the encoder | §2.3: forward-looking exogenous information is legitimately available at forecast time |
| Calendar | recovered from `time_idx` by spectral analysis (base period `m`), encoded as Fourier pairs | §2.8 / Q2.8: no timestamps are given, the cycle must be *recovered* |
| Validation | **rolling-origin** chronological blocks of 168, many origins, held out with a gap | a single 168-block score is far too noisy to make decisions on (§2.5) |
| Seeds | every claim is averaged over `N_SEEDS` runs with mean ± std | §2.4.3 |
| Ablation | with / without the optional file, across seeds, same split | §2.4.4 |
| Cost | `d_model = 32`, 2 encoder / 1 decoder layer, ~3·10⁴ params, single-digit epochs | `P` and `E` are penalised in the score (§2.7) |

**Attribution.** The Autoformer blocks below are a from-scratch re-implementation following
Wu et al. (2021), *Autoformer: Decomposition Transformers with Auto-Correlation for Long-Term Series
Forecasting*, §§3.1–3.2, and reuse the `SeriesDecomposition` / `delay_scores` / `aggregate_delays`
formulation developed in **Question 1** of this assignment (Eqs. 3–6). Deviations from the reference
are listed where they occur.

**Notebook layout.** One function (or one class) per cell; the final section runs the pipeline.

## 1. Environment, configuration and reproducibility

In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from dataclasses import dataclass, field, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# CPU-bound environment: use every physical core, and keep BLAS from oversubscribing.
torch.set_num_threads(max(1, (os.cpu_count() or 4) // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("Data")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
pd.set_option("display.width", 160)

print(f"torch {torch.__version__} | device={DEVICE} | threads={torch.get_num_threads()}")

In [ ]:
@dataclass(frozen=True)
class Config:
    """Every knob of the pipeline in one place, so runs are reproducible and diffable."""

    # --- forecasting protocol (fixed by the task) --------------------------------
    pred_len: int = 168                 # §2.4.2 — hard requirement
    n_total: int = 43_824               # full series length incl. the hidden block
    first_test_idx: int = 43_657

    # --- window geometry ---------------------------------------------------------
    seq_len: int = 336                  # encoder context = 2 base cycles of 168
    label_len: int = 168                # decoder warm-start taken from the context

    # --- target handling ---------------------------------------------------------
    target_transform: str = "sqrt"      # {"identity", "sqrt", "log1p"}; selected in 16
    smearing: bool = True               # Duan correction for the convex back-transform

    # --- features ----------------------------------------------------------------
    use_external: bool = True           # master switch for the §2.4.4 ablation
    use_external_deltas: bool = True     # first differences of the continuous externals
    use_calendar: bool = True

    # --- model (kept deliberately small: P is penalised in the score) ------------
    d_model: int = 32
    n_heads: int = 4
    e_layers: int = 2
    d_layers: int = 1
    d_ff: int = 64
    moving_avg: int = 25                # odd kernel of the internal decomposition
    autocorr_factor: int = 1            # k = factor * ceil(log(L)) aggregated delays
    dropout: float = 0.0                # see note below: underfitting is the binding constraint

    # --- optimisation -------------------------------------------------------------
    epochs: int = 8
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    train_stride: int = 3               # subsample overlapping windows -> cheaper epochs
    warmup_frac: float = 0.05

    # --- validation protocol -------------------------------------------------------
    val_span: int = 168 * 26            # ~half a year of rolling origins
    val_stride: int = 24                # one origin per base cycle
    calib_frac: float = 0.25            # earliest quarter of held-out blocks calibrates only
    gap: int = 168                      # buffer between train targets and held-out contexts

    seeds: tuple[int, ...] = (0, 1, 2)

    @property
    def dec_len(self) -> int:
        return self.label_len + self.pred_len


CFG = Config()
CFG

In [ ]:
def set_seed(seed: int) -> None:
    """Seed every RNG the pipeline touches."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 2. Data loading

Three files, all on the same strictly regular integer clock. `optional_external_data.csv` is
168 rows longer than the target history — it *covers the hidden block*, which is the whole point
of §2.3.

In [ ]:
def load_raw(data_dir: Path = DATA_DIR) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read the three supplied files and assert the structural guarantees of §2.2."""
    train = pd.read_csv(data_dir / "student_train.csv")
    test = pd.read_csv(data_dir / "student_test.csv")
    external = pd.read_csv(data_dir / "optional_external_data.csv")

    assert train["time_idx"].is_monotonic_increasing
    assert np.array_equal(train["time_idx"].to_numpy(), np.arange(1, len(train) + 1)), "non-uniform grid"
    assert train["value"].notna().all(), "train is documented to be complete"
    assert len(test) == CFG.pred_len and test["time_idx"].iloc[0] == CFG.first_test_idx
    assert len(external) == CFG.n_total == len(train) + CFG.pred_len
    assert external["time_idx"].is_monotonic_increasing and external.notna().all().all()
    return train, test, external

## 3. Exploratory analysis

Two questions have to be answered before any modelling:

1. **What is the sampling interval / the base cycle?** The timestamps were stripped (Q2.8), so the
   period has to be *recovered* from the data — a periodogram and the autocorrelation function do that.
2. **How far does autocorrelation actually reach?** If it dies well inside 168 steps, an endogenous
   model cannot reach the horizon and the exogenous file becomes the dominant source of signal.

In [ ]:
def autocorrelation(x: np.ndarray, max_lag: int) -> np.ndarray:
    """Unbiased-in-the-mean circular ACF via FFT (O(n log n), no explicit lag loop)."""
    z = x - x.mean()
    n_fft = 1 << int(np.ceil(np.log2(2 * len(z))))
    spec = np.fft.rfft(z, n_fft)
    acf = np.fft.irfft(spec * np.conj(spec), n_fft)[: max_lag + 1]
    return acf / acf[0]


def dominant_periods(x: np.ndarray, top: int = 6, min_period: int = 3) -> pd.DataFrame:
    """Rank candidate periods by periodogram power (Wiener-Khinchin, fully vectorised)."""
    z = x - x.mean()
    power = np.abs(np.fft.rfft(z)) ** 2
    freq = np.fft.rfftfreq(len(z), d=1.0)
    valid = freq > 0
    period = 1.0 / freq[valid]
    power = power[valid]
    keep = period >= min_period
    order = np.argsort(power[keep])[::-1][:top]
    return pd.DataFrame({"period": np.round(period[keep][order], 2),
                         "relative_power": power[keep][order] / power[keep].sum()})

In [ ]:
def explore_series(values: np.ndarray, external: pd.DataFrame, max_lag: int = 400) -> dict:
    """Six-panel overview + the numbers that drive the design decisions."""
    acf = autocorrelation(values, max_lag)
    periods = dominant_periods(values, top=8)

    fig, ax = plt.subplots(2, 3, figsize=(14, 6.2))

    ax[0, 0].plot(values, lw=0.25)
    ax[0, 0].set(title="Full history", xlabel="time_idx", ylabel="value")

    ax[0, 1].plot(values[-168 * 6:], lw=0.8)
    ax[0, 1].set(title="Last 6 horizons (1008 steps)", xlabel="steps before end")

    ax[0, 2].hist(values, bins=120, color="tab:purple")
    ax[0, 2].set(title=f"Marginal (skew={pd.Series(values).skew():.2f})", yscale="log")

    ax[1, 0].plot(acf, lw=0.9)
    ax[1, 0].axhline(0, color="k", lw=0.6)
    ax[1, 0].axvline(CFG.pred_len, color="tab:red", ls="--", lw=0.9, label="horizon = 168")
    ax[1, 0].set(title="Autocorrelation", xlabel="lag"); ax[1, 0].legend()

    # Periodogram restricted to the interpretable band.
    z = values - values.mean()
    power = np.abs(np.fft.rfft(z)) ** 2
    freq = np.fft.rfftfreq(len(z))
    band = (freq > 1 / 400) & (freq < 0.5)
    ax[1, 1].semilogy(1 / freq[band], power[band], lw=0.5)
    ax[1, 1].set(title="Periodogram", xlabel="period (steps)", xscale="log")

    corr = pd.DataFrame(external.iloc[: len(values), 1:]).assign(target=np.log1p(values)).corr()["target"][:-1]
    corr.plot.barh(ax=ax[1, 2], color=np.where(corr > 0, "tab:red", "tab:blue"))
    ax[1, 2].set(title="corr(external, log1p target)")

    fig.tight_layout(); plt.show()

    base = int(round(periods.loc[periods["period"].between(8, 60), "period"].iloc[0]))
    summary = {
        "base_period": base,
        "acf_at_1": acf[1], "acf_at_24": acf[24], "acf_at_168": acf[168],
        "first_lag_below_0.1": int(np.argmax(acf < 0.1)),
        "top_periods": periods,
    }
    print(f"recovered base period     : {base} steps")
    print(f"ACF(1)={acf[1]:.3f}  ACF({base})={acf[base]:.3f}  ACF(168)={acf[168]:.3f}")
    print(f"ACF drops below 0.1 at lag: {summary['first_lag_below_0.1']}")
    display(periods)
    return summary

## 4. Feature engineering

**Target.** The marginal is non-negative, right-skewed and heavy-tailed. `log1p` turns the
multiplicative spikes into additive ones so that an MSE objective is not spent almost entirely on a
few extreme hours; the forecast is mapped back with `expm1` plus a **Duan smearing** factor, which
is the correct inversion when the loss is squared error in log space.

**Covariates.** All ten optional variables are measured on the target's clock *and extend over the
horizon*, so they are **known-future** covariates: the right place for them is the decoder stamp,
not just the encoder. Treatment:

* `feature_A/B/C` — continuous, roughly symmetric → standardised as-is.
* `feature_D/E/F` — continuous but strongly right-skewed accumulations → `log1p` then standardised.
* `feature_G..J` — mutually exclusive binaries, already a clean one-hot → left untouched.
* **first differences** of the six continuous variables: a level alone does not say whether the
  regime is *arriving* or *leaving*, and the transitions are what move the target.

**Calendar.** No timestamps exist, only an integer index (Q2.8). Having recovered the base period
`m` spectrally, position within the cycle, within `7m`, and within the annual cycle are encoded as
sine/cosine pairs — continuous, bounded, and free of the discontinuity that a raw counter has.

In [ ]:
CONTINUOUS = ["feature_A", "feature_B", "feature_C"]
SKEWED = ["feature_D", "feature_E", "feature_F"]
BINARY = ["feature_G", "feature_H", "feature_I", "feature_J"]


def build_covariates(external: pd.DataFrame, base_period: int, cfg: Config) -> tuple[np.ndarray, list[str]]:
    """Return the [n_total, n_cov] covariate matrix (raw, unstandardised) and its column names."""
    blocks, names = [], []

    if cfg.use_external:
        cont = external[CONTINUOUS].to_numpy(np.float32)
        skew = np.log1p(np.clip(external[SKEWED].to_numpy(np.float32), 0, None))
        binr = external[BINARY].to_numpy(np.float32)
        blocks += [cont, skew, binr]
        names += CONTINUOUS + [f"log1p_{c}" for c in SKEWED] + BINARY

        if cfg.use_external_deltas:
            level = np.concatenate([cont, skew], axis=1)
            delta = np.diff(level, axis=0, prepend=level[:1])
            blocks.append(delta)
            names += [f"d_{c}" for c in CONTINUOUS + SKEWED]

    if cfg.use_calendar:
        t = external["time_idx"].to_numpy(np.float32) - 1.0
        cycles = {"day": base_period, "week": 7 * base_period, "year": 365.25 * base_period}
        harmonics = [(name, p, h) for name, p in cycles.items() for h in (1, 2)]
        fourier = np.column_stack(
            [f(2 * np.pi * h * t / p) for _, p, h in harmonics for f in (np.sin, np.cos)]
        ).astype(np.float32)
        blocks.append(fourier)
        names += [f"{fn}{h}_{name}" for name, _, h in harmonics for fn in ("sin", "cos")]

    if not blocks:  # pure-endogenous ablation still needs a (zero-width-safe) stamp
        return np.zeros((len(external), 0), np.float32), []
    return np.concatenate(blocks, axis=1), names

In [ ]:
class Standardiser:
    """Column-wise z-scoring fitted on a training slice only (no look-ahead)."""

    def __init__(self, matrix: np.ndarray, fit_end: int, skip: np.ndarray | None = None):
        train = matrix[:fit_end]
        self.mean = train.mean(0)
        self.std = train.std(0)
        self.std[self.std < 1e-6] = 1.0
        if skip is not None and skip.any():        # leave already-normalised one-hots alone
            self.mean[skip] = 0.0
            self.std[skip] = 1.0

    def __call__(self, matrix: np.ndarray) -> np.ndarray:
        return ((matrix - self.mean) / self.std).astype(np.float32)

In [ ]:
class TargetTransform:
    """Variance-stabilising map plus its (smearing-corrected) inverse.

    Training minimises MSE on g(y) standardised; `inverse` must therefore undo

        y_hat = g^{-1}(sigma * u + mu) * s

    where `s` is Duan's smearing factor. For a convex `g^{-1}` such as `expm1`, the naive inversion
    is biased low exactly where the series spikes, which is precisely where squared error is paid.
    For `identity` the inverse is linear, there is no such bias, and no correction is applied.
    """

    _FORWARD = {"identity": lambda y: y,
                "sqrt": np.sqrt,
                "log1p": np.log1p}
    _INVERSE = {"identity": lambda u: u,
                "sqrt": lambda u: np.square(np.clip(u, 0, None)),
                "log1p": lambda u: np.expm1(np.clip(u, None, 12.0))}

    # A bias correction should be a nudge; anything outside this range means the fit is degenerate
    # (an untrained model, say) rather than mildly biased, so it is refused rather than applied.
    SMEAR_BOUNDS = (0.5, 2.0)

    def __init__(self, values: np.ndarray, fit_end: int, kind: str):
        if kind not in self._FORWARD:
            raise ValueError(f"unknown transform {kind!r}")
        self.kind = kind
        self.latent = self._FORWARD[kind](values.astype(np.float64))
        train = self.latent[:fit_end]
        self.mean = float(train.mean())
        self.std = float(train.std()) or 1.0
        self.smear = 1.0

    @property
    def corrects_bias(self) -> bool:
        """Only a non-linear inverse introduces the bias a smearing factor exists to remove."""
        return self.kind != "identity"

    def forward(self, values: np.ndarray) -> np.ndarray:
        return ((self._FORWARD[self.kind](values) - self.mean) / self.std).astype(np.float32)

    @property
    def normalised(self) -> np.ndarray:
        return ((self.latent - self.mean) / self.std).astype(np.float32)

    def _inverse_raw(self, normalised: np.ndarray) -> np.ndarray:
        latent = np.asarray(normalised, np.float64) * self.std + self.mean
        return np.clip(self._INVERSE[self.kind](latent), 0.0, None)

    def inverse(self, normalised: np.ndarray) -> np.ndarray:
        return self._inverse_raw(normalised) * self.smear

    def fit_smearing(self, pred_norm: np.ndarray, truth: np.ndarray) -> float:
        """Least-squares scalar on the *uncorrected* inverse; the RMSE-optimal choice of `s`."""
        base = self._inverse_raw(pred_norm)
        denom = float((base ** 2).sum())
        factor = float((base * truth).sum() / denom) if denom > 0 else 1.0
        low, high = self.SMEAR_BOUNDS
        self.smear = factor if low <= factor <= high else 1.0
        return self.smear

## 5. Window construction and the chronological split

A sample is one `(encoder window, decoder window)` pair anchored at an origin `o`:

```
encoder    [o - seq_len, o)                 target history + covariates
decoder    [o - label_len, o + pred_len)    label_len warm-start, then 168 unknown steps
```

Windows are never materialised as a dense tensor (that would be ~1 GB); batches are gathered on the
fly with one vectorised fancy-index per batch — no per-sample Python loop.

**Split.** Strictly chronological. The last `val_span` steps supply `val_span / val_stride`
rolling validation origins; a `gap` of one full horizon separates the last training *target* from the
first validation *context*, so no training label can leak into a validation input.

In [ ]:
def split_origins(n_history: int, cfg: Config) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Origins `o` such that the decoder target is [o, o + pred_len).

    Returns `(train, calib, val)`. The validation region is split chronologically: the earliest
    `calib_frac` of its blocks only ever fit the scalar back-transform correction, the rest are the
    blocks every reported number is measured on. Nothing is both calibrated on and reported on.
    """
    first = cfg.seq_len                                    # need a full context behind us
    last = n_history - cfg.pred_len                        # need a full target ahead of us

    held_start = max(first, n_history - cfg.val_span)
    held = np.arange(held_start, last + 1, cfg.val_stride)
    cut = max(1, int(round(cfg.calib_frac * len(held))))
    calib, val = held[:cut], held[cut:]

    # Training targets must end before the earliest held-out *context* starts.
    train_target_end = (held[0] - cfg.seq_len) - cfg.gap
    train = np.arange(first, train_target_end - cfg.pred_len + 1, cfg.train_stride)
    assert len(train) and len(calib) and len(val), "empty split"
    return train, calib, val

In [ ]:
def gather_batch(origins: np.ndarray, target: np.ndarray, cov: np.ndarray, cfg: Config):
    """Vectorised window gather for a batch of origins.

    Returns numpy arrays
        x_enc      [B, seq_len, 1]        past target
        mark_enc   [B, seq_len, C]        past covariates
        mark_dec   [B, dec_len, C]        warm-start + known-future covariates
        y          [B, pred_len]          target over the horizon (normalised space)
    """
    enc_off = np.arange(-cfg.seq_len, 0)
    dec_off = np.arange(-cfg.label_len, cfg.pred_len)
    tgt_off = np.arange(0, cfg.pred_len)

    enc_idx = origins[:, None] + enc_off          # [B, seq_len]
    dec_idx = origins[:, None] + dec_off          # [B, dec_len]
    tgt_idx = origins[:, None] + tgt_off          # [B, pred_len]

    x_enc = target[enc_idx][..., None]
    mark_enc = cov[enc_idx]
    mark_dec = cov[dec_idx]
    y = target[tgt_idx]
    return x_enc, mark_enc, mark_dec, y


def to_tensors(batch, device=DEVICE):
    return tuple(torch.from_numpy(np.ascontiguousarray(a)).to(device) for a in batch)

## 6. The Autoformer

Three components, each in its own cell, following Wu et al. (2021) §§3.1–3.2.

### 6.1 Series decomposition (§3.1)
`SeriesDecomp` is a centred moving average with **replicate padding**, so trend and seasonal have the
same length as the input and reconstruct it exactly (`seasonal + trend = x`). It is applied
*progressively*: once on the encoder input, and again after **every** residual sub-layer, so each
block hands the next one a remainder rather than a stream into which it has re-injected slow
structure. This is the mechanism required by §2.4.1(a).

In [ ]:
class SeriesDecomp(nn.Module):
    """x -> (seasonal, trend) with a centred moving average of odd width `kernel`.

    Same operator as Question 1's `SeriesDecomposition`: replicate-pad by `kernel//2` on both ends,
    so `seasonal + trend == x` exactly and both keep the input length. Parameter-free.

    Implemented as a difference of cumulative sums rather than `avg_pool1d`, which makes it O(L)
    instead of O(L*kernel); the decomposition runs 8 times per forward pass, so this matters.
    """

    def __init__(self, kernel: int):
        super().__init__()
        if kernel < 1 or kernel % 2 == 0:
            raise ValueError("kernel must be positive and odd")
        self.kernel = kernel
        self.pad = kernel // 2

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x: [B, L, C]; pad along time with the nearest endpoint value.
        padded = F.pad(x.transpose(1, 2), (self.pad, self.pad), mode="replicate")
        cumulative = F.pad(torch.cumsum(padded, dim=-1), (1, 0))
        trend = ((cumulative[..., self.kernel:] - cumulative[..., :-self.kernel])
                 / self.kernel).transpose(1, 2)
        return x - trend, trend

### 6.2 Auto-Correlation (§3.2)

`delay_scores` is the series-wise correlation
$R(\tau)=\sum_t q_t\,k_{(t-\tau)\bmod L}$ for **all** $\tau$ at once, obtained as
`irfft(rfft(q) · conj(rfft(k)))` — $O(L\log L)$ instead of $O(L^2)$.

`aggregate_delays` is Eq. (6): keep the top-$k$ delays, soft-max their scores into weights, and sum
the correspondingly **rolled** value sequences. This is dependency discovery *at the level of
periods*, not point-wise dot-product attention — the mechanism required by §2.4.1(b).

Deviations from the reference implementation, deliberate and stated:

* delays are selected **per example** (as in Question 1 / Eq. 5–6) rather than shared across the
  training batch — a batch-shared lag is an artefact of the original code, not of the method, and a
  per-example lag is what inference does anyway, so train and test now behave identically;
* $\tau = 0$ is excluded (it is the trivial self-match and carries no delay information);
* the whole top-$k$ aggregation is vectorised with a single `gather`, with no Python loop over $k$.

In [ ]:
def delay_scores(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """R(tau) for every tau in O(L log L).  q, k: [B, H, E, L] -> [B, H, E, L]."""
    length = q.shape[-1]
    q_fft = torch.fft.rfft(q.float(), n=length, dim=-1)
    k_fft = torch.fft.rfft(k.float(), n=length, dim=-1)
    return torch.fft.irfft(q_fft * torch.conj(k_fft), n=length, dim=-1)


def aggregate_delays(values: torch.Tensor, delays: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    """Eq. (6): z_t = sum_j alpha_j * v_{(t - tau_j) mod L}.

    values  [B, H, E, L]
    delays  [B, K]  integer lags, selected per example
    weights [B, K]  non-negative, summing to one

    Scattering the K weights into a dense length-L kernel `alpha` turns Eq. (6) into a *circular
    convolution* of `v` with `alpha`, because

        (alpha (*) v)_t = sum_s alpha_s v_{(t - s) mod L}

    and `alpha_s` is zero for every `s` outside the selected delays. The convolution theorem then
    evaluates all L output positions at once in O(L log L), with no Python loop over K and no
    materialised [B, H, E, K, L] gather. `scatter_add_` (not `scatter_`) keeps the result correct
    if a lag were ever selected twice. The equivalence is checked against the literal definition in
    the verification cell below.
    """
    length = values.shape[-1]
    alpha = values.new_zeros(values.shape[0], length).scatter_add_(1, delays, weights)
    spectrum = (torch.fft.rfft(values.float(), n=length, dim=-1)
                * torch.fft.rfft(alpha.float(), n=length, dim=-1)[:, None, None, :])
    return torch.fft.irfft(spectrum, n=length, dim=-1)

In [ ]:
class AutoCorrelation(nn.Module):
    """Period-based dependency discovery + time-delay aggregation (Autoformer §3.2)."""

    def __init__(self, factor: int = 1, attention_dropout: float = 0.0):
        super().__init__()
        self.factor = factor
        self.dropout = nn.Dropout(attention_dropout)

    def forward(self, queries, keys, values):
        # queries/keys/values: [B, L, H, E]; keys/values may be shorter or longer than queries.
        b, lq, heads, width = queries.shape
        lk = keys.shape[1]

        if lk < lq:                                   # pad with a wrap of itself (cross-attention)
            reps = math.ceil(lq / lk)
            keys = keys.repeat(1, reps, 1, 1)[:, :lq]
            values = values.repeat(1, reps, 1, 1)[:, :lq]
        else:
            keys, values = keys[:, :lq], values[:, :lq]

        q = queries.permute(0, 2, 3, 1)               # [B, H, E, L]
        kk = keys.permute(0, 2, 3, 1)
        vv = values.permute(0, 2, 3, 1)

        scores = delay_scores(q, kk)                  # [B, H, E, L]
        mean_scores = scores.mean(dim=(1, 2))         # [B, L] — one lag budget per example
        mean_scores[:, 0] = -torch.inf               # drop the trivial tau = 0 self-match

        top_k = max(1, min(lq - 1, int(self.factor * math.ceil(math.log(lq)))))
        top_scores, delays = torch.topk(mean_scores, top_k, dim=-1)
        weights = self.dropout(torch.softmax(top_scores, dim=-1))

        out = aggregate_delays(vv, delays, weights)   # [B, H, E, L]
        return out.permute(0, 3, 1, 2).contiguous()   # [B, L, H, E]

#### Verification of the two supplied mechanisms

Both are easy to get subtly wrong (an off-by-one in the shift direction silently costs accuracy),
so each is checked against its literal definition on a small worked example before any training.

In [ ]:
def verify_autocorrelation(length: int = 16, seed: int = 0) -> None:
    """Check `delay_scores` and `aggregate_delays` against their O(L^2) definitions."""
    torch.manual_seed(seed)
    q, k = torch.randn(2, 2, 3, length), torch.randn(2, 2, 3, length)

    # Eq. (4): R(tau) = sum_t q_t k_{(t - tau) mod L}
    fast = delay_scores(q, k)
    direct = torch.stack([(q * torch.roll(k, shifts=tau, dims=-1)).sum(-1)
                          for tau in range(length)], dim=-1)
    assert torch.allclose(fast, direct, atol=1e-4), (fast - direct).abs().max()

    # Eq. (6): z_t = sum_j alpha_j v_{(t - tau_j) mod L}
    b, heads, width, top_k = 3, 2, 4, 3
    values = torch.randn(b, heads, width, length)
    delays = torch.stack([torch.randperm(length - 1)[:top_k] + 1 for _ in range(b)])
    weights = torch.softmax(torch.randn(b, top_k), dim=-1)

    got = aggregate_delays(values, delays, weights)
    want = torch.stack([sum(weights[i, j] * torch.roll(values[i], int(delays[i, j]), dims=-1)
                            for j in range(top_k)) for i in range(b)])
    assert torch.allclose(got, want, atol=1e-4), (got - want).abs().max()

    # tau = 0 with unit weight must be the identity, and the weights must be a simplex.
    identity = aggregate_delays(values, torch.zeros(b, 1, dtype=torch.long), torch.ones(b, 1))
    assert torch.allclose(identity, values, atol=1e-5)

    # Decomposition must reconstruct exactly, and leave a constant signal untouched.
    decomp = SeriesDecomp(25)
    x = torch.randn(4, 96, 3)
    seasonal, trend = decomp(x)
    assert torch.allclose(seasonal + trend, x, atol=1e-5)
    const = torch.full((2, 50, 1), 3.0)
    s_c, t_c = decomp(const)
    assert torch.allclose(t_c, const, atol=1e-5) and s_c.abs().max() < 1e-5

    print("verified: delay_scores == Eq.(4) | aggregate_delays == Eq.(6) | "
          "SeriesDecomp reconstructs and is replicate-padded")


verify_autocorrelation()

In [ ]:
class AutoCorrelationLayer(nn.Module):
    """Projection wrapper: d_model -> heads x d_head, correlate, project back."""

    def __init__(self, d_model: int, n_heads: int, factor: int, dropout: float):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.inner = AutoCorrelation(factor, dropout)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, queries, keys, values):
        b, lq, _ = queries.shape
        lk = keys.shape[1]
        h = self.n_heads
        q = self.q_proj(queries).view(b, lq, h, -1)
        k = self.k_proj(keys).view(b, lk, h, -1)
        v = self.v_proj(values).view(b, lk, h, -1)
        out = self.inner(q, k, v).view(b, lq, -1)
        return self.out_proj(out)

### 6.3 Embedding, encoder, decoder

* **Value embedding** — a circular 1-D convolution over the series channel: it gives every position a
  short local receptive field before any mixing happens, at a trivial parameter cost.
* **Stamp embedding** — one linear map of the covariate vector (externals + calendar Fourier pairs).
  The encoder stamp carries the *past* covariates, the decoder stamp the *warm-start plus the
  known-future* ones. This is where §2.3's forward-looking information enters (Q2.7).
* **Positional encoding** — fixed sinusoids, zero parameters.
* **Encoder** — `e_layers` × (AutoCorrelation → decomp → feed-forward → decomp). Only the seasonal
  stream propagates; the trend each sub-layer removes is discarded here.
* **Decoder** — `d_layers` × (self AutoCorrelation → decomp → cross AutoCorrelation → decomp →
  feed-forward → decomp), and every discarded trend piece is **accumulated** into a running trend
  which is projected and added back at the end. The seasonal and trend streams are initialised from
  the decomposition of the encoder input (seasonal → zeros, trend → context mean over the horizon),
  so the model starts from a sensible level and only has to learn the deviation.

In [ ]:
class DataEmbedding(nn.Module):
    """value conv + covariate stamp + fixed sinusoidal position."""

    def __init__(self, c_in: int, n_cov: int, d_model: int, dropout: float, max_len: int = 5000):
        super().__init__()
        self.value = nn.Conv1d(c_in, d_model, kernel_size=3, padding=1,
                               padding_mode="circular", bias=False)
        nn.init.kaiming_normal_(self.value.weight, mode="fan_in", nonlinearity="leaky_relu")
        self.stamp = nn.Linear(n_cov, d_model, bias=False) if n_cov else None
        self.register_buffer("position", self._sinusoids(max_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)

    @staticmethod
    def _sinusoids(max_len: int, d_model: int) -> torch.Tensor:
        pos = torch.arange(max_len, dtype=torch.float32)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe[None]

    def forward(self, x: torch.Tensor, mark: torch.Tensor | None) -> torch.Tensor:
        out = self.value(x.transpose(1, 2)).transpose(1, 2) + self.position[:, : x.shape[1]]
        if self.stamp is not None and mark is not None and mark.shape[-1]:
            out = out + self.stamp(mark)
        return self.drop(out)

In [ ]:
class EncoderLayer(nn.Module):
    """AutoCorrelation + feed-forward, each followed by decomposition instead of LayerNorm-only."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.attn = AutoCorrelationLayer(cfg.d_model, cfg.n_heads, cfg.autocorr_factor, cfg.dropout)
        self.conv1 = nn.Conv1d(cfg.d_model, cfg.d_ff, 1, bias=False)
        self.conv2 = nn.Conv1d(cfg.d_ff, cfg.d_model, 1, bias=False)
        self.decomp1 = SeriesDecomp(cfg.moving_avg)
        self.decomp2 = SeriesDecomp(cfg.moving_avg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop(self.attn(x, x, x))
        x, _ = self.decomp1(x)
        y = self.conv2(self.drop(F.gelu(self.conv1(x.transpose(1, 2))))).transpose(1, 2)
        x, _ = self.decomp2(x + self.drop(y))
        return x


class Encoder(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.layers = nn.ModuleList(EncoderLayer(cfg) for _ in range(cfg.e_layers))
        self.norm = nn.LayerNorm(cfg.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

In [ ]:
class DecoderLayer(nn.Module):
    """Self + cross AutoCorrelation; every removed trend piece is handed back to the caller."""

    def __init__(self, cfg: Config, c_out: int):
        super().__init__()
        self.self_attn = AutoCorrelationLayer(cfg.d_model, cfg.n_heads, cfg.autocorr_factor, cfg.dropout)
        self.cross_attn = AutoCorrelationLayer(cfg.d_model, cfg.n_heads, cfg.autocorr_factor, cfg.dropout)
        self.conv1 = nn.Conv1d(cfg.d_model, cfg.d_ff, 1, bias=False)
        self.conv2 = nn.Conv1d(cfg.d_ff, cfg.d_model, 1, bias=False)
        self.decomp1 = SeriesDecomp(cfg.moving_avg)
        self.decomp2 = SeriesDecomp(cfg.moving_avg)
        self.decomp3 = SeriesDecomp(cfg.moving_avg)
        self.trend_proj = nn.Conv1d(cfg.d_model, c_out, 3, padding=1,
                                    padding_mode="circular", bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, memory: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = x + self.drop(self.self_attn(x, x, x))
        x, t1 = self.decomp1(x)
        x = x + self.drop(self.cross_attn(x, memory, memory))
        x, t2 = self.decomp2(x)
        y = self.conv2(self.drop(F.gelu(self.conv1(x.transpose(1, 2))))).transpose(1, 2)
        x, t3 = self.decomp3(x + self.drop(y))
        trend = self.trend_proj((t1 + t2 + t3).transpose(1, 2)).transpose(1, 2)
        return x, trend


class Decoder(nn.Module):
    def __init__(self, cfg: Config, c_out: int):
        super().__init__()
        self.layers = nn.ModuleList(DecoderLayer(cfg, c_out) for _ in range(cfg.d_layers))
        self.norm = nn.LayerNorm(cfg.d_model)
        self.projection = nn.Linear(cfg.d_model, c_out, bias=True)

    def forward(self, x, memory, trend):
        for layer in self.layers:
            x, residual_trend = layer(x, memory)
            trend = trend + residual_trend
        return self.projection(self.norm(x)) + trend

In [ ]:
class Autoformer(nn.Module):
    """Decomposition encoder-decoder with Auto-Correlation (Wu et al., 2021)."""

    def __init__(self, cfg: Config, n_cov: int, c_in: int = 1, c_out: int = 1):
        super().__init__()
        self.cfg = cfg
        self.decomp = SeriesDecomp(cfg.moving_avg)
        self.enc_embedding = DataEmbedding(c_in, n_cov, cfg.d_model, cfg.dropout)
        self.dec_embedding = DataEmbedding(c_in, n_cov, cfg.d_model, cfg.dropout)
        self.encoder = Encoder(cfg)
        self.decoder = Decoder(cfg, c_out)

    def forward(self, x_enc, mark_enc, mark_dec):
        cfg = self.cfg
        # --- decoder initialisation (Autoformer §3.1): seasonal -> 0, trend -> context mean -------
        seasonal_ctx, trend_ctx = self.decomp(x_enc)
        level = x_enc.mean(dim=1, keepdim=True).expand(-1, cfg.pred_len, -1)
        trend_init = torch.cat([trend_ctx[:, -cfg.label_len:], level], dim=1)
        seasonal_init = torch.cat(
            [seasonal_ctx[:, -cfg.label_len:], torch.zeros_like(level)], dim=1)

        memory = self.encoder(self.enc_embedding(x_enc, mark_enc))
        dec_in = self.dec_embedding(seasonal_init, mark_dec)
        out = self.decoder(dec_in, memory, trend_init)
        return out[:, -cfg.pred_len:, 0]          # [B, pred_len]


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 7. Metrics

All three §2.6 metrics are tracked, always **in raw units**, never in the transformed space.
They disagree by construction: RMSE is driven by the few largest hours, MAE by the typical hour, and
sMAPE by the *small* ones (its denominator shrinks with the level). Ranking uses RMSE, so RMSE is
what model selection optimises — but a configuration that wins on RMSE while collapsing sMAPE is
usually just predicting the mean, and the three columns make that visible.

In [ ]:
def metrics(pred: np.ndarray, truth: np.ndarray) -> dict[str, float]:
    """MAE / RMSE / sMAPE over flattened raw-unit arrays (§2.6)."""
    pred = np.asarray(pred, np.float64).ravel()
    truth = np.asarray(truth, np.float64).ravel()
    err = pred - truth
    denom = np.abs(truth) + np.abs(pred)
    smape = np.where(denom > 0, 2.0 * np.abs(err) / np.where(denom > 0, denom, 1.0), 0.0)
    return {"MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
            "sMAPE": float(100.0 * smape.mean())}


def summarise(frames: list[dict], keys=("MAE", "RMSE", "sMAPE")) -> pd.Series:
    """mean +/- std across seeds, as a flat labelled series."""
    table = pd.DataFrame(frames)
    out = {}
    for k in keys:
        out[f"{k}_mean"] = table[k].mean()
        out[f"{k}_std"] = table[k].std(ddof=1) if len(table) > 1 else 0.0
    return pd.Series(out)

## 8. Training and evaluation

A plain, short schedule: AdamW, linear warm-up into a cosine decay, gradient clipping, MSE on the
standardised transformed target. The epoch budget is small on purpose — `E` is penalised (§2.7) and
this series has a low signal-to-noise ratio at a 168-step horizon, so a long schedule mostly buys
over-fitting.

**Why `dropout = 0`.** The model has ~3·10⁴ parameters against >10⁴ training windows and a
single-digit epoch budget, so the binding constraint here is *under*-fitting, not over-fitting;
weight decay already supplies the small amount of shrinkage that is wanted. It is also not free: on
CPU the Bernoulli RNG inside `nn.Dropout` costs roughly 40 % of a training step. Regularisation
strength is a `Config` field, so this is a measured default and not an assumption.

In [ ]:
def iterate_minibatches(origins: np.ndarray, batch_size: int, rng: np.random.Generator | None):
    """Yield index slices; shuffled when `rng` is given, chronological otherwise."""
    order = rng.permutation(len(origins)) if rng is not None else np.arange(len(origins))
    for start in range(0, len(order), batch_size):
        yield origins[order[start:start + batch_size]]

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, origins: np.ndarray, target: np.ndarray, cov: np.ndarray,
            cfg: Config, batch_size: int = 256) -> np.ndarray:
    """Normalised-space forecasts for every origin, shape [n_origins, pred_len]."""
    model.eval()
    chunks = [model(*to_tensors(gather_batch(idx, target, cov, cfg)[:3])).cpu().numpy()
              for idx in iterate_minibatches(origins, batch_size, None)]
    return np.concatenate(chunks, axis=0)

In [ ]:
def train_model(cfg: Config, target: np.ndarray, cov: np.ndarray, train_origins: np.ndarray,
                seed: int, epochs: int | None = None, verbose: bool = False) -> tuple[nn.Module, dict]:
    """Fit one Autoformer. Returns the model and a small history dict."""
    set_seed(seed)
    epochs = cfg.epochs if epochs is None else epochs
    model = Autoformer(cfg, n_cov=cov.shape[1]).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps_per_epoch = math.ceil(len(train_origins) / cfg.batch_size)
    total_steps = max(1, steps_per_epoch * epochs)
    warmup = max(1, int(cfg.warmup_frac * total_steps))

    def lr_at(step: int) -> float:
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    rng = np.random.default_rng(seed)
    history = {"train_loss": [], "epoch_seconds": [], "epochs": epochs,
               "params": count_parameters(model)}

    for epoch in range(epochs):
        model.train()
        tic, running, seen = time.perf_counter(), 0.0, 0
        for idx in iterate_minibatches(train_origins, cfg.batch_size, rng):
            x_enc, mark_enc, mark_dec, y = to_tensors(gather_batch(idx, target, cov, cfg))
            loss = F.mse_loss(model(x_enc, mark_enc, mark_dec), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            sched.step()
            running += loss.item() * len(idx)
            seen += len(idx)
        history["train_loss"].append(running / seen)
        history["epoch_seconds"].append(time.perf_counter() - tic)
        if verbose:
            print(f"  epoch {epoch + 1:>2}/{epochs}  loss={history['train_loss'][-1]:.4f}"
                  f"  ({history['epoch_seconds'][-1]:.1f}s)")
    return model, history

In [ ]:
def evaluate(model: nn.Module, origins: np.ndarray, target_norm: np.ndarray, cov: np.ndarray,
             raw_values: np.ndarray, transform: TargetTransform, cfg: Config) -> tuple[dict, np.ndarray, np.ndarray]:
    """Rolling-origin evaluation in raw units over the given origins."""
    pred_norm = predict(model, origins, target_norm, cov, cfg)
    truth = raw_values[origins[:, None] + np.arange(cfg.pred_len)[None, :]]
    return metrics(transform.inverse(pred_norm), truth), transform.inverse(pred_norm), truth


def calibrate_transform(model: nn.Module, bundle: dict, cfg: Config) -> float:
    """Fit the scalar back-transform correction on the calibration blocks only."""
    transform = bundle["transform"]
    transform.smear = 1.0
    if not (cfg.smearing and transform.corrects_bias):
        return 1.0
    origins = bundle["calib_origins"]
    pred_norm = predict(model, origins, bundle["target_norm"], bundle["cov"], cfg)
    truth = bundle["raw"][origins[:, None] + np.arange(cfg.pred_len)[None, :]]
    return transform.fit_smearing(pred_norm, truth)

## 9. Reference baselines

Without these, a neural RMSE is an uninterpretable number. Three cheap, honest references evaluated
on *exactly* the same rolling origins:

* **persistence** — repeat the last observed value (optimal under a random walk);
* **seasonal naive** — repeat the last full base cycle (optimal if the cycle is everything);
* **cycle-profile climatology** — the training mean per position in the base cycle, which is the best
  a model that knows only "where in the cycle am I" can do.

A model that cannot clear all three has learned nothing worth 30k parameters.

In [ ]:
def baseline_forecasts(origins: np.ndarray, raw: np.ndarray, base_period: int,
                       fit_end: int, cfg: Config) -> dict[str, np.ndarray]:
    """Vectorised baselines, each [n_origins, pred_len] in raw units."""
    h = np.arange(cfg.pred_len)
    persistence = np.repeat(raw[origins - 1][:, None], cfg.pred_len, axis=1)

    season_idx = origins[:, None] + h[None, :] - cfg.pred_len
    seasonal_naive = raw[season_idx]

    phase = np.arange(fit_end) % base_period
    profile = np.bincount(phase, weights=raw[:fit_end], minlength=base_period) / np.bincount(phase, minlength=base_period)
    climatology = profile[(origins[:, None] + h[None, :]) % base_period]
    return {"persistence": persistence,
            "seasonal naive": seasonal_naive,
            "cycle climatology": climatology}

## 10. Multi-seed experiment driver

Every reported comparison goes through this one function, so "with externals" and "without externals"
differ *only* by the flag under test: same origins, same seeds, same budget (§2.4.3).

In [ ]:
def run_experiment(cfg: Config, bundle: dict, label: str, epochs: int | None = None,
                   seeds: tuple[int, ...] | None = None, verbose: bool = False) -> dict:
    """Train one model per seed; calibrate on the calibration blocks, score on the validation ones."""
    seeds = cfg.seeds if seeds is None else seeds
    rows, preds, histories, smears = [], [], [], []
    for seed in seeds:
        tic = time.perf_counter()
        model, hist = train_model(cfg, bundle["target_norm"], bundle["cov"],
                                  bundle["train_origins"], seed, epochs, verbose)
        smears.append(calibrate_transform(model, bundle, cfg))
        scores, pred, truth = evaluate(model, bundle["val_origins"], bundle["target_norm"],
                                       bundle["cov"], bundle["raw"], bundle["transform"], cfg)
        scores["seed"] = seed
        scores["fit_seconds"] = time.perf_counter() - tic
        rows.append(scores)
        preds.append(pred)
        histories.append(hist)
        print(f"[{label}] seed {seed}: RMSE={scores['RMSE']:7.3f}  MAE={scores['MAE']:7.3f}"
              f"  sMAPE={scores['sMAPE']:6.2f}  ({scores['fit_seconds']:.0f}s)")

    ensemble = metrics(np.mean(preds, axis=0), truth)
    summary = summarise(rows)
    print(f"[{label}] mean RMSE={summary['RMSE_mean']:.3f} +/- {summary['RMSE_std']:.3f}"
          f" | {len(seeds)}-seed ensemble RMSE={ensemble['RMSE']:.3f}"
          f" | P={histories[0]['params']:,} E={histories[0]['epochs']}")
    return {"label": label, "rows": rows, "summary": summary, "ensemble": ensemble,
            "histories": histories, "preds": np.array(preds), "truth": truth,
            "params": histories[0]["params"], "epochs": histories[0]["epochs"],
            "smear": float(np.mean(smears))}


def screen(configs: dict[str, tuple[Config, int]], train_df: pd.DataFrame, external_df: pd.DataFrame,
           base_period: int, seeds: tuple[int, ...]) -> tuple[pd.DataFrame, dict]:
    """Cheap single/low-seed screen over a dict of {label: (config, epochs)}.

    Screening narrows the design space; it never settles a *claim*. Anything reported as a result
    is re-run over the full seed set (§2.4.3).
    """
    rows, raw = {}, {}
    for label, (cfg, epochs) in configs.items():
        bundle = prepare(cfg, train_df, external_df, base_period)
        result = run_experiment(cfg, bundle, label, epochs=epochs, seeds=seeds)
        raw[label] = result
        rows[label] = {"RMSE": result["summary"]["RMSE_mean"], "MAE": result["summary"]["MAE_mean"],
                       "sMAPE": result["summary"]["sMAPE_mean"], "P": result["params"],
                       "E": result["epochs"],
                       "seconds": float(np.mean([r["fit_seconds"] for r in result["rows"]]))}
    return pd.DataFrame(rows).T, raw

## 11. Assemble the dataset

One function builds everything a run needs from a `Config`, so the ablation is a one-line change.
Every statistic (target mean/std, covariate mean/std) is fitted on the **training region only**.

In [ ]:
def prepare(cfg: Config, train: pd.DataFrame, external: pd.DataFrame, base_period: int) -> dict:
    """Return the full bundle of arrays/objects needed to train and evaluate under `cfg`."""
    raw = train["value"].to_numpy(np.float64)
    n_history = len(raw)

    train_origins, calib_origins, val_origins = split_origins(n_history, cfg)
    fit_end = int(calib_origins[0] - cfg.seq_len)        # nothing at or beyond this is ever fitted

    transform = TargetTransform(raw, fit_end, cfg.target_transform)
    target_norm = transform.normalised.astype(np.float32)
    # The hidden block has no target; pad with the last value so window gathers stay in range.
    target_full = np.concatenate([target_norm, np.repeat(target_norm[-1], cfg.pred_len)])

    cov_raw, cov_names = build_covariates(external, base_period, cfg)
    if cov_raw.shape[1]:
        skip = np.isin(cov_names, BINARY)
        cov = Standardiser(cov_raw, fit_end, skip)(cov_raw)
    else:
        cov = cov_raw

    return {"raw": raw, "target_norm": target_full, "cov": cov, "cov_names": cov_names,
            "transform": transform, "train_origins": train_origins,
            "calib_origins": calib_origins, "val_origins": val_origins,
            "fit_end": fit_end, "n_history": n_history, "base_period": base_period}

---
# Pipeline

Everything above is definitions; everything below runs.

## 12. Load and explore

In [ ]:
train_df, test_df, external_df = load_raw()
print(f"train {train_df.shape} | test {test_df.shape} | external {external_df.shape}")
display(train_df.head(3))
display(external_df.head(3))
values = train_df["value"].to_numpy(np.float64)
print(train_df["value"].describe().round(2).to_dict())

In [ ]:
eda = explore_series(values, external_df)
BASE_PERIOD = eda["base_period"]

**Read-out.**

* The periodogram and ACF recover a base cycle of `BASE_PERIOD` steps (plus a weaker harmonic at
  `7 × BASE_PERIOD`), which pins the sampling interval well enough to build calendar features — the
  series is hourly, the horizon of 168 steps is one week.
* The ACF collapses below 0.1 long before lag 168. **Autoregression alone cannot reach this
  horizon**; past the first day or so the conditional mean of an endogenous model degenerates
  towards climatology. That is the single most important fact for the design, and it is exactly why
  the known-future external block is worth testing seriously.
* The marginal is strongly right-skewed with a long upper tail — hence the `log1p` modelling space.
* Several externals correlate meaningfully with `log1p(value)` (notably the skewed accumulation
  `feature_D` and the binary `feature_H`), so there is signal to extract.

## 13. Build the dataset and check the split

In [ ]:
bundle = prepare(CFG, train_df, external_df, BASE_PERIOD)
print(f"covariates ({len(bundle['cov_names'])}): {bundle['cov_names']}")
print(f"train origins : {len(bundle['train_origins']):>6,}  "
      f"targets span [{bundle['train_origins'][0]}, {bundle['train_origins'][-1] + CFG.pred_len})")
print(f"calib origins : {len(bundle['calib_origins']):>6,}  "
      f"targets span [{bundle['calib_origins'][0]}, {bundle['calib_origins'][-1] + CFG.pred_len})")
print(f"val   origins : {len(bundle['val_origins']):>6,}  "
      f"targets span [{bundle['val_origins'][0]}, {bundle['val_origins'][-1] + CFG.pred_len})")
print(f"fit_end (no statistic uses data at/after this index): {bundle['fit_end']:,}")
assert bundle["train_origins"][-1] + CFG.pred_len + CFG.gap <= bundle["calib_origins"][0] - CFG.seq_len
assert bundle["calib_origins"][-1] < bundle["val_origins"][0]
print("leakage checks passed: train targets precede held-out contexts by >= gap; "
      "calibration blocks precede the reported blocks")

In [ ]:
def plot_split(bundle: dict, cfg: Config) -> None:
    fig, ax = plt.subplots(figsize=(13, 2.8))
    ax.plot(bundle["raw"], lw=0.2, color="0.4")
    spans = [
        (cfg.seq_len, bundle["train_origins"][-1] + cfg.pred_len, "tab:blue", 0.12,
         f"train targets ({len(bundle['train_origins']):,} origins)"),
        (bundle["calib_origins"][0], bundle["calib_origins"][-1] + cfg.pred_len, "tab:green", 0.25,
         f"calibration ({len(bundle['calib_origins'])} blocks)"),
        (bundle["val_origins"][0], bundle["val_origins"][-1] + cfg.pred_len, "tab:orange", 0.25,
         f"validation ({len(bundle['val_origins'])} blocks of 168)"),
        (bundle["n_history"], cfg.n_total, "tab:red", 0.35, "hidden block"),
    ]
    for start, stop, colour, alpha, label in spans:
        ax.axvspan(start, stop, color=colour, alpha=alpha, label=label)
    ax.set(title="Chronological split", xlabel="time_idx", ylabel="value")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout(); plt.show()


plot_split(bundle, CFG)

## 14. Baselines on the validation origins

In [ ]:
base_preds = baseline_forecasts(bundle["val_origins"], bundle["raw"], BASE_PERIOD,
                                bundle["fit_end"], CFG)
val_truth = bundle["raw"][bundle["val_origins"][:, None] + np.arange(CFG.pred_len)[None, :]]
baseline_table = pd.DataFrame({name: metrics(p, val_truth) for name, p in base_preds.items()}).T
display(baseline_table.round(3))
BASELINE_RMSE = float(baseline_table["RMSE"].min())
print(f"best baseline RMSE on the validation blocks: {BASELINE_RMSE:.3f}")

## 15. Sanity check: one short run

Before spending the seed budget, confirm the model trains, learns something, and fits in a sane
amount of wall-clock time.

In [ ]:
_smoke_cfg = replace(CFG, epochs=1, train_stride=12)
_smoke_bundle = prepare(_smoke_cfg, train_df, external_df, BASE_PERIOD)
_model, _hist = train_model(_smoke_cfg, _smoke_bundle["target_norm"], _smoke_bundle["cov"],
                            _smoke_bundle["train_origins"], seed=0, verbose=True)
calibrate_transform(_model, _smoke_bundle, _smoke_cfg)
_scores, _, _ = evaluate(_model, _smoke_bundle["val_origins"], _smoke_bundle["target_norm"],
                         _smoke_bundle["cov"], _smoke_bundle["raw"], _smoke_bundle["transform"],
                         _smoke_cfg)
print(f"smoke run: params={count_parameters(_model):,} | "
      f"{_hist['epoch_seconds'][0]:.1f}s/epoch at stride {_smoke_cfg.train_stride} | {_scores}")
del _model

## 16. Design screen

Two choices are decided here *before* the headline experiments, on a single seed, because screening
only has to narrow the space — it does not settle a claim. Everything that ends up in the report is
re-run over the full seed set in §17.

**Target space.** `identity`, `sqrt` and `log1p` encode three different beliefs about where the error
lives: constant variance, variance growing with the level, and multiplicative error. Since the
ranking metric is RMSE in raw units, the question is empirical.

**Width.** `d_model` is the main lever on `P`, which is penalised directly. The screen reports RMSE
*and* `P`, so the choice is made on the cost frontier rather than on accuracy alone.

In [ ]:
SCREEN_SEEDS = (0,)
SCREEN_EPOCHS = 8

transform_screen, _ = screen(
    {kind: (replace(CFG, target_transform=kind), SCREEN_EPOCHS)
     for kind in ("identity", "sqrt", "log1p")},
    train_df, external_df, BASE_PERIOD, SCREEN_SEEDS)
display(transform_screen.round(3))
BEST_TRANSFORM = transform_screen["RMSE"].idxmin()
print(f"selected target space: {BEST_TRANSFORM!r}")

In [ ]:
width_screen, _ = screen(
    {f"d_model={d}": (replace(CFG, target_transform=BEST_TRANSFORM, d_model=d, d_ff=2 * d),
                      SCREEN_EPOCHS)
     for d in (16, 32, 64)},
    train_df, external_df, BASE_PERIOD, SCREEN_SEEDS)
width_screen["RMSE_per_1k_params"] = width_screen["RMSE"] / (width_screen["P"] / 1000)
display(width_screen.round(3))

### Costing a design choice

`Score = RMSE + α·P + β·E` and the coefficients are not published, so every size/cost decision in
this notebook goes through one declared assumption rather than through ad-hoc judgement. The
frontier plot shows what the decision is sensitive to, and the same function is used for both
remaining cost decisions — model width and whether to ensemble — so they cannot disagree.

In [ ]:
# Declared assumptions, not measurements: the true alpha/beta are withheld (§2.7). They are stated
# once, used everywhere a cost decision is made, and their sensitivity is visible in the frontier.
ALPHA_ASSUMED = 2e-5      # 1 RMSE point is worth ~50k parameters
BETA_ASSUMED = 5e-2       # 1 RMSE point is worth ~20 epochs


def leaderboard_score(rmse: float, params: int, epochs: int) -> float:
    """The §2.7 ranking score under the declared penalty assumptions (lower is better)."""
    return rmse + ALPHA_ASSUMED * params + BETA_ASSUMED * epochs

In [ ]:
def plot_cost_frontier(table: pd.DataFrame) -> None:
    """RMSE against P, with the implied break-even alpha between neighbouring sizes."""
    fig, ax = plt.subplots(figsize=(5.6, 3.4))
    ax.plot(table["P"], table["RMSE"], "o-", color="tab:blue")
    for name, row in table.iterrows():
        ax.annotate(name, (row["P"], row["RMSE"]), textcoords="offset points",
                    xytext=(6, 5), fontsize=8)
    ax.set(xlabel="trainable parameters P", ylabel="validation RMSE",
           title="Accuracy / cost frontier")
    fig.tight_layout(); plt.show()

    ordered = table.sort_values("P")
    break_even = (-ordered["RMSE"].diff() / ordered["P"].diff()).dropna()
    print("break-even alpha when stepping up in size (RMSE bought per extra parameter):")
    for name, value in break_even.items():
        verdict = "never worth it" if value <= 0 else f"worth it only if alpha < {value:.2e}"
        print(f"  -> {name}: {verdict}")


plot_cost_frontier(width_screen)

In [ ]:
penalised = width_screen.apply(
    lambda r: leaderboard_score(r["RMSE"], int(r["P"]), int(r["E"])), axis=1)
display(penalised.round(3).rename("RMSE + alpha*P + beta*E"))
BEST_D_MODEL = int(penalised.idxmin().split("=")[1])
BEST_CFG = replace(CFG, target_transform=BEST_TRANSFORM, d_model=BEST_D_MODEL, d_ff=2 * BEST_D_MODEL)
print(f"selected d_model = {BEST_D_MODEL} "
      f"(alpha = {ALPHA_ASSUMED:g}, beta = {BETA_ASSUMED:g})")

## 17. §2.4.4 ablation — does the optional external file help?

Identical configuration, identical origins, identical seeds; the only difference is
`use_external`. Three variants, so the answer is not just yes/no but *which part* of the file
matters (§2.3 asks exactly that):

1. **endogenous only** — calendar Fourier features, no external file at all;
2. **external levels** — the ten supplied variables, known across the horizon;
3. **external levels + differences** — plus the first differences of the six continuous ones.

In [ ]:
ABLATION = {
    "endogenous only":   replace(BEST_CFG, use_external=False, use_external_deltas=False),
    "external levels":   replace(BEST_CFG, use_external=True,  use_external_deltas=False),
    "external + deltas": replace(BEST_CFG, use_external=True,  use_external_deltas=True),
}

results = {}
for name, cfg in ABLATION.items():
    b = prepare(cfg, train_df, external_df, BASE_PERIOD)
    results[name] = run_experiment(cfg, b, name, epochs=SCREEN_EPOCHS)

In [ ]:
ablation_table = pd.DataFrame({
    name: {
        "params": r["params"],
        "epochs": r["epochs"],
        "RMSE": r["summary"]["RMSE_mean"],
        "RMSE_sd": r["summary"]["RMSE_std"],
        "MAE": r["summary"]["MAE_mean"],
        "sMAPE": r["summary"]["sMAPE_mean"],
        "ensemble_RMSE": r["ensemble"]["RMSE"],
    } for name, r in results.items()
}).T
ablation_table.loc["best baseline"] = {"params": 0, "epochs": 0, "RMSE": BASELINE_RMSE,
                                       "RMSE_sd": 0.0,
                                       "MAE": float(baseline_table["MAE"].min()),
                                       "sMAPE": float(baseline_table["sMAPE"].min()),
                                       "ensemble_RMSE": BASELINE_RMSE}
display(ablation_table.round(3))

best_name = ablation_table.drop(index="best baseline")["RMSE"].idxmin()
delta = ablation_table.loc["endogenous only", "RMSE"] - ablation_table.loc[best_name, "RMSE"]
pooled_sd = float(np.hypot(ablation_table.loc["endogenous only", "RMSE_sd"],
                           ablation_table.loc[best_name, "RMSE_sd"]))
print(f"best variant: {best_name!r}")
print(f"the external file is worth {delta:+.3f} RMSE against endogenous-only"
      + (f" ({delta / pooled_sd:+.1f} pooled seed sd)" if pooled_sd > 1e-9 else ""))
FINAL_CFG = ABLATION[best_name]

In [ ]:
def plot_ablation(results: dict, baseline_rmse: float) -> None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
    names = list(results)
    means = [results[n]["summary"]["RMSE_mean"] for n in names]
    sds = [results[n]["summary"]["RMSE_std"] for n in names]
    ax[0].bar(range(len(names)), means, yerr=sds, capsize=4, color="tab:blue", alpha=0.85)
    ax[0].axhline(baseline_rmse, color="tab:red", ls="--", lw=1, label="best baseline")
    ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels(names, rotation=12, ha="right")
    ax[0].set(ylabel="validation RMSE", title=f"Ablation ({len(CFG.seeds)} seeds, mean +/- sd)")
    ax[0].legend(fontsize=8)

    for n in names:
        for h in results[n]["histories"]:
            ax[1].plot(range(1, len(h["train_loss"]) + 1), h["train_loss"], lw=0.9, alpha=0.7)
    ax[1].set(xlabel="epoch", ylabel="train MSE (normalised space)", title="Training curves")
    fig.tight_layout(); plt.show()


plot_ablation(results, BASELINE_RMSE)

## 18. Epoch budget

`E` is penalised, so the schedule is chosen rather than inherited. Validation RMSE is measured at
several budgets on the winning configuration, across seeds, and the smallest budget that is not
meaningfully worse than the best is selected — a one-standard-deviation rule, which is the right
level of scepticism given how noisy this series is.

In [ ]:
def epoch_sweep(cfg: Config, bundle: dict, budgets: tuple[int, ...],
                seeds: tuple[int, ...]) -> tuple[pd.DataFrame, dict]:
    """Validation RMSE vs epoch budget, averaged over seeds.

    The per-budget results are returned as well, so the selected budget does not have to be
    re-trained: the run that chose it *is* the run the report quotes.
    """
    rows, results = [], {}
    for e in budgets:
        res = run_experiment(cfg, bundle, f"epochs={e}", epochs=e, seeds=seeds)
        results[e] = res
        rows.append({"epochs": e,
                     "RMSE": res["summary"]["RMSE_mean"],
                     "RMSE_sd": res["summary"]["RMSE_std"],
                     "MAE": res["summary"]["MAE_mean"],
                     "sMAPE": res["summary"]["sMAPE_mean"],
                     "seconds_per_fit": float(np.mean([r["fit_seconds"] for r in res["rows"]]))})
    return pd.DataFrame(rows).set_index("epochs"), results


final_bundle = prepare(FINAL_CFG, train_df, external_df, BASE_PERIOD)
sweep, sweep_results = epoch_sweep(FINAL_CFG, final_bundle, budgets=(4, 8, 12), seeds=CFG.seeds)
display(sweep.round(3))

_floor = sweep["RMSE"].min() + sweep.loc[sweep["RMSE"].idxmin(), "RMSE_sd"]
CHOSEN_EPOCHS = int(sweep.index[sweep["RMSE"] <= _floor][0])
print(f"chosen epoch budget (smallest within 1 sd of the best): {CHOSEN_EPOCHS}")

In [ ]:
FINAL_RESULT = sweep_results[CHOSEN_EPOCHS]
FINAL_SMEAR = FINAL_RESULT["smear"]
print(f"selected model : {best_name}")
print(f"  transform    : {FINAL_CFG.target_transform}")
print(f"  d_model      : {FINAL_CFG.d_model}  (P = {FINAL_RESULT['params']:,} per model)")
print(f"  epochs       : {CHOSEN_EPOCHS}")
print(f"  smearing     : {FINAL_SMEAR:.4f}")
print(f"  validation   : RMSE {FINAL_RESULT['summary']['RMSE_mean']:.3f}"
      f" +/- {FINAL_RESULT['summary']['RMSE_std']:.3f} over {len(CFG.seeds)} seeds,"
      f" {len(final_bundle['val_origins'])} blocks")
print(f"  best baseline: RMSE {BASELINE_RMSE:.3f}")

## 19. Qualitative check

Validation RMSE is a scalar; these two views say *how* the forecast fails, which is what the report
has to explain. The horizon profile in particular shows where the endogenous signal runs out.

In [ ]:
def plot_forecasts(result: dict, origins: np.ndarray) -> None:
    pred, truth = result["preds"].mean(axis=0), result["truth"]
    rmse_per_block = np.sqrt(((pred - truth) ** 2).mean(axis=1))
    order = np.argsort(rmse_per_block)
    picks = [int(order[0]), int(order[len(order) // 2]), int(order[-1])]
    titles = ["best block", "median block", "worst block"]

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.1))
    for a, i, t in zip(ax, picks, titles):
        a.plot(truth[i], lw=1.1, color="k", label="actual")
        a.plot(pred[i], lw=1.3, color="tab:red", label="Autoformer")
        a.set(title=f"{t} (origin {origins[i]}, RMSE={rmse_per_block[i]:.1f})", xlabel="horizon step")
    ax[0].set_ylabel("value"); ax[0].legend(fontsize=8)
    fig.tight_layout(); plt.show()


def plot_horizon_error(results: dict, base_preds: dict, truth: np.ndarray) -> None:
    fig, ax = plt.subplots(figsize=(8, 3.2))
    for name, r in results.items():
        ax.plot(np.sqrt(((r["preds"].mean(0) - truth) ** 2).mean(axis=0)), lw=1.2, label=name)
    for name, p in base_preds.items():
        ax.plot(np.sqrt(((p - truth) ** 2).mean(axis=0)), lw=0.9, ls="--", alpha=0.65, label=name)
    ax.set(xlabel="horizon step", ylabel="RMSE", title="Error growth across the 168-step horizon")
    ax.legend(fontsize=7, ncol=2); fig.tight_layout(); plt.show()


plot_forecasts(FINAL_RESULT, final_bundle["val_origins"])
plot_horizon_error({**results, "selected": FINAL_RESULT}, base_preds, val_truth)

## 20. Final fit and submission

The selected configuration is refitted on the **entire** supplied history — the held-out region is
genuinely informative about the forecast origin's regime, and holding it out at submission time
would waste the most recent and most relevant data.

`P` and `E` are computed from the artefacts themselves (§2.7), never typed in by hand:

* `P` = `sum(p.numel() for p in model.parameters() if p.requires_grad)`, summed over every model that
  contributes to the submitted forecast;
* `E` = total epochs actually spent producing those forecasts. The sweeps above are *model
  selection*, not production of the submitted numbers, so they are not counted; the figure below is
  the budget of the run that actually generated the 168 values.

The smearing factor is **not** re-estimated here; it is carried over from the calibration blocks,
which the reported validation numbers never used.

In [ ]:
def final_origins(cfg: Config, n_history: int) -> np.ndarray:
    """All origins usable for the final fit, including the most recent ones."""
    return np.arange(cfg.seq_len, n_history - cfg.pred_len + 1, cfg.train_stride)


def fit_final_and_forecast(cfg: Config, train_df: pd.DataFrame, external_df: pd.DataFrame,
                           base_period: int, epochs: int, seeds: tuple[int, ...],
                           smear: float) -> dict:
    """Refit on all available history and forecast the hidden 168-step block."""
    bundle = prepare(cfg, train_df, external_df, base_period)
    n_history = bundle["n_history"]
    origins = final_origins(cfg, n_history)
    forecast_origin = np.array([n_history])      # targets = [n_history, n_history + 168)
    bundle["transform"].smear = smear

    members, total_params, total_epochs = [], 0, 0
    for seed in seeds:
        model, hist = train_model(cfg, bundle["target_norm"], bundle["cov"], origins, seed,
                                  epochs, verbose=True)
        members.append(predict(model, forecast_origin, bundle["target_norm"], bundle["cov"], cfg)[0])
        total_params += hist["params"]
        total_epochs += hist["epochs"]
        print(f"  seed {seed} done ({sum(hist['epoch_seconds']):.0f}s)")

    forecast = bundle["transform"].inverse(np.mean(members, axis=0))
    return {"forecast": forecast, "P": total_params, "E": total_epochs,
            "n_members": len(seeds), "bundle": bundle, "train_origins": origins, "smear": smear}

In [ ]:
# Ensembling is allowed but paid for: k members cost k*P and k*E. The decision uses the same
# declared penalty as the width choice, and additionally refuses to act on a gain smaller than the
# seed-to-seed spread -- a difference that small is noise, not an improvement (§2.5).
_k = len(CFG.seeds)
_single_rmse = FINAL_RESULT["summary"]["RMSE_mean"]
_ens_rmse = FINAL_RESULT["ensemble"]["RMSE"]
_seed_sd = FINAL_RESULT["summary"]["RMSE_std"]
_P, _E = FINAL_RESULT["params"], CHOSEN_EPOCHS

comparison = pd.DataFrame({
    "single model": {"RMSE": _single_rmse, "P": _P, "E": _E,
                     "score": leaderboard_score(_single_rmse, _P, _E)},
    f"{_k}-seed ensemble": {"RMSE": _ens_rmse, "P": _k * _P, "E": _k * _E,
                            "score": leaderboard_score(_ens_rmse, _k * _P, _k * _E)},
}).T
display(comparison.round(3))

_worth_it = (comparison.loc[f"{_k}-seed ensemble", "score"] < comparison.loc["single model", "score"]
             and (_single_rmse - _ens_rmse) > _seed_sd)
SUBMIT_SEEDS = CFG.seeds if _worth_it else (CFG.seeds[0],)
print(f"ensemble gain = {_single_rmse - _ens_rmse:+.3f} RMSE against a seed sd of {_seed_sd:.3f}")
print(f"submitting with {len(SUBMIT_SEEDS)} model(s): seeds {SUBMIT_SEEDS}")

In [ ]:
final = fit_final_and_forecast(FINAL_CFG, train_df, external_df, BASE_PERIOD,
                               epochs=CHOSEN_EPOCHS, seeds=SUBMIT_SEEDS, smear=FINAL_SMEAR)
forecast = final["forecast"]

assert forecast.shape == (CFG.pred_len,), forecast.shape
assert np.isfinite(forecast).all() and (forecast >= 0).all()
print(f"forecast: min={forecast.min():.2f}  mean={forecast.mean():.2f}  max={forecast.max():.2f}")
print(f"declared P = {final['P']:,} trainable parameters")
print(f"declared E = {final['E']} epochs")

In [ ]:
def plot_submission(history: np.ndarray, forecast: np.ndarray, cfg: Config, context: int = 672) -> None:
    fig, ax = plt.subplots(figsize=(12, 3.2))
    h = history[-context:]
    ax.plot(np.arange(len(history) - len(h), len(history)) + 1, h, lw=0.8, color="k", label="history")
    ax.plot(np.arange(cfg.first_test_idx, cfg.first_test_idx + cfg.pred_len), forecast,
            lw=1.5, color="tab:red", label="Autoformer forecast")
    ax.axvline(len(history) + 0.5, color="tab:blue", ls="--", lw=1)
    ax.set(xlabel="time_idx", ylabel="value", title="Submitted 168-step forecast")
    ax.legend(fontsize=8); fig.tight_layout(); plt.show()


plot_submission(values, forecast, CFG)

In [ ]:
def write_submission(forecast: np.ndarray, test_df: pd.DataFrame, meta: dict,
                     out_dir: Path = ARTIFACT_DIR) -> str:
    """Write the paste-ready prediction string plus a reproducible record of P and E."""
    assert len(forecast) == len(test_df) == CFG.pred_len
    payload = ", ".join(f"{v:.4f}" for v in np.asarray(forecast, float))

    (out_dir / "predictions.txt").write_text(payload, encoding="utf-8")
    test_df.assign(value=forecast).to_csv(out_dir / "student_test_filled.csv", index=False)
    (out_dir / "submission_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"wrote {out_dir/'predictions.txt'} ({len(forecast)} values)")
    return payload


meta = {
    "trainable_parameters_P": int(final["P"]),
    "training_epochs_E": int(final["E"]),
    "ensemble_members": final["n_members"],
    "seeds": list(SUBMIT_SEEDS),
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in FINAL_CFG.__dict__.items()},
    "base_period": int(BASE_PERIOD),
    "smearing_factor": float(final["smear"]),
    "declared_penalties": {"alpha_assumed": ALPHA_ASSUMED, "beta_assumed": BETA_ASSUMED},
    "validation": {
        "n_blocks": int(len(final_bundle["val_origins"])),
        "n_calibration_blocks": int(len(final_bundle["calib_origins"])),
        "RMSE_mean": float(FINAL_RESULT["summary"]["RMSE_mean"]),
        "RMSE_std": float(FINAL_RESULT["summary"]["RMSE_std"]),
        "MAE_mean": float(FINAL_RESULT["summary"]["MAE_mean"]),
        "sMAPE_mean": float(FINAL_RESULT["summary"]["sMAPE_mean"]),
        "best_baseline_RMSE": float(BASELINE_RMSE),
        "endogenous_only_RMSE": float(ablation_table.loc["endogenous only", "RMSE"]),
    },
}
predictions_string = write_submission(forecast, test_df, meta)
print(predictions_string[:200], "...")
print(json.dumps(meta["validation"], indent=2))

## 21. Summary — what to carry into the report

Numbers below are printed by the cells above; this section says what each one is evidence *for*.

* **Recovered structure (Q2.8).** The base period is recovered spectrally from `time_idx` alone, and
  the 168-step horizon is exactly `7 ×` that cycle. No calendar was supplied, so the Fourier stamp is
  the only way the model knows where in the cycle it stands. The recovery is trustworthy only while
  the cycle length is stable — which the periodogram supports over the full history.
* **The horizon is beyond the endogenous memory.** The ACF is effectively dead well before lag 168,
  and in the horizon-error plot the endogenous-only model flattens out at the climatology level
  within roughly the first base cycle. That is a property of the *series*, not of the architecture,
  and it is what makes the exogenous question decisive rather than cosmetic.
* **The optional file (§2.4.4).** Quantified in §17 across seeds on one fixed rolling split, with the
  levels-vs-differences distinction isolated. Report the effect size *and* the seed spread: a
  difference smaller than the spread is not a result (§2.4.3).
* **Where the gain comes from (Q2.7).** These covariates are known *across the horizon*, so they
  enter the decoder stamp. A covariate known only up to the origin could only have helped through the
  encoder, and would have decayed with the same memory as the target itself.
* **Cost (§2.7).** `P` and `E` in `artifacts/submission_meta.json` are computed from the objects that
  produced the submitted numbers. The frontier in §16 shows what extra width would have cost and what
  it would have bought, so the size choice is defended rather than assumed.
* **Metric disagreement (§2.6).** RMSE is set almost entirely by a handful of spike steps, MAE by the
  typical step, and sMAPE by the *low* ones, since its denominator shrinks with the level. A model
  tuned on RMSE is tuned on the spikes — which is also why the variance-stabilising transform and its
  bias-corrected inverse were chosen on evidence rather than by habit.